# Notebook 01: Foundations

Fundamental building blocks of a transformer:
- Matrix operations with NumPy
- Linear (fully-connected) layer with forward **and backward** pass
- Activation functions (ReLU, Softmax) with gradients
- Numerical gradient checking

Everything is implemented from scratch using only NumPy.

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Key Matrix Operations

Transformers rely heavily on matrix multiplications. The core operations are batched dense linear maps.

In [ ]:
# Matrix multiplication: the core operation of neural networks
# A linear layer computes: y = x @ W + b

x = np.random.randn(2, 3)  # batch of 2 vectors, each dim 3
W = np.random.randn(3, 4)  # weight matrix projecting dim 3 -> dim 4
b = np.random.randn(4)     # bias vector

y = x @ W + b
print(f"Input shape:  {x.shape}")
print(f"Weight shape: {W.shape}")
print(f"Output shape: {y.shape}")
print(f"\nOutput:\n{y}")

## 2. Linear Layer (with Backpropagation)

A linear layer computes $y = xW + b$.

For backpropagation we need gradients:
- $\frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial y}$
- $\frac{\partial L}{\partial b} = \sum \frac{\partial L}{\partial y}$ (sum over batch)
- $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot W^T$

In [ ]:
class Linear:
    """Fully-connected layer with forward and backward pass."""
    
    def __init__(self, in_features, out_features):
        # Xavier/Glorot initialization for stable training
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)
        
        # Gradient storage
        self.dW = None
        self.db = None
        
        # Cache for backward pass
        self.x = None
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, dout):
        # dout: gradient of loss w.r.t. output, shape (batch, out_features)
        self.dW = self.x.reshape(-1, self.x.shape[-1]).T @ dout.reshape(-1, dout.shape[-1])
        self.db = dout.reshape(-1, dout.shape[-1]).sum(axis=0)
        dx = dout @ self.W.T
        return dx

# Test it
layer = Linear(3, 4)
x = np.random.randn(2, 3)
y = layer.forward(x)
print(f"Forward: input {x.shape} -> output {y.shape}")

# Simulate a gradient flowing back
dout = np.random.randn(2, 4)
dx = layer.backward(dout)
print(f"Backward: dout {dout.shape} -> dx {dx.shape}")
print(f"Weight gradient shape: {layer.dW.shape}")
print(f"Bias gradient shape:   {layer.db.shape}")

## 3. Activation Functions

### ReLU
$$\text{ReLU}(x) = \max(0, x)$$
$$\frac{\partial}{\partial x} \text{ReLU}(x) = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{otherwise} \end{cases}$$

### Softmax
$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

We use softmax combined with cross-entropy loss for numerical stability.

In [ ]:
class ReLU:
    """ReLU activation with forward and backward pass."""
    
    def __init__(self):
        self.mask = None
    
    def forward(self, x):
        self.mask = (x > 0).astype(float)
        return x * self.mask
    
    def backward(self, dout):
        return dout * self.mask


def softmax(x):
    """Numerically stable softmax."""
    # Subtract max for numerical stability (prevents overflow)
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def cross_entropy_loss(logits, targets):
    """Cross-entropy loss with softmax.
    
    Args:
        logits: raw scores, shape (N, C) where C = num classes
        targets: integer class labels, shape (N,)
    
    Returns:
        loss: scalar
        dlogits: gradient w.r.t. logits, shape (N, C)
    """
    N = logits.shape[0]
    probs = softmax(logits)
    
    # Loss: -log(probability of correct class)
    log_probs = -np.log(probs[np.arange(N), targets] + 1e-9)
    loss = np.mean(log_probs)
    
    # Gradient of softmax + cross-entropy combined
    dlogits = probs.copy()
    dlogits[np.arange(N), targets] -= 1
    dlogits /= N
    
    return loss, dlogits

# Test ReLU
relu = ReLU()
x = np.array([-2, -1, 0, 1, 2], dtype=float)
print(f"ReLU input:  {x}")
print(f"ReLU output: {relu.forward(x)}")
print(f"ReLU grad:   {relu.backward(np.ones_like(x))}")

# Test softmax
logits = np.array([[2.0, 1.0, 0.1]])
print(f"\nSoftmax input:  {logits}")
print(f"Softmax output: {softmax(logits)}")
print(f"Sum of probs:   {softmax(logits).sum():.6f}")

# Test cross-entropy
logits = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
targets = np.array([0, 1])  # correct classes
loss, dlogits = cross_entropy_loss(logits, targets)
print(f"\nCross-entropy loss: {loss:.4f}")
print(f"Gradient shape: {dlogits.shape}")

## 4. Numerical Gradient Checking

To verify our backpropagation is correct, we compare analytical gradients (from backward pass) with numerical gradients computed via finite differences:

$$\frac{\partial L}{\partial w_i} \approx \frac{L(w_i + \epsilon) - L(w_i - \epsilon)}{2\epsilon}$$

If the relative error is < 1e-5, our gradients are correct.

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    """Compute numerical gradient of f at x using central differences."""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = x[idx]
        
        x[idx] = old_val + eps
        fp = f(x)
        
        x[idx] = old_val - eps
        fm = f(x)
        
        grad[idx] = (fp - fm) / (2 * eps)
        x[idx] = old_val
        it.iternext()
    return grad


def relative_error(a, b):
    """Compute relative error between two arrays."""
    return np.max(np.abs(a - b) / (np.maximum(np.abs(a) + np.abs(b), 1e-8)))


# Gradient check for Linear layer
np.random.seed(42)
layer = Linear(3, 4)
x = np.random.randn(2, 3)
targets = np.array([1, 3])

# Forward + backward
logits = layer.forward(x)
loss, dlogits = cross_entropy_loss(logits, targets)
dx = layer.backward(dlogits)

# Numerical gradient for W
def loss_fn_W(W):
    layer.W = W
    logits = layer.forward(x)
    loss, _ = cross_entropy_loss(logits, targets)
    return loss

num_grad_W = numerical_gradient(loss_fn_W, layer.W.copy())
err_W = relative_error(layer.dW, num_grad_W)
print(f"W gradient relative error: {err_W:.2e} {'PASS' if err_W < 1e-5 else 'FAIL'}")

# Numerical gradient for b
def loss_fn_b(b):
    layer.b = b
    logits = layer.forward(x)
    loss, _ = cross_entropy_loss(logits, targets)
    return loss

num_grad_b = numerical_gradient(loss_fn_b, layer.b.copy())
err_b = relative_error(layer.db, num_grad_b)
print(f"b gradient relative error: {err_b:.2e} {'PASS' if err_b < 1e-5 else 'FAIL'}")

# Numerical gradient for input x
def loss_fn_x(x_in):
    logits = layer.forward(x_in)
    loss, _ = cross_entropy_loss(logits, targets)
    return loss

num_grad_x = numerical_gradient(loss_fn_x, x.copy())
err_x = relative_error(dx, num_grad_x)
print(f"x gradient relative error: {err_x:.2e} {'PASS' if err_x < 1e-5 else 'FAIL'}")

## 5. Simple Training Demo

Train a single linear layer on a simple mapping to verify the training loop works.

In [ ]:
import matplotlib.pyplot as plt

# Simple task: classify 2D points into 3 classes
np.random.seed(42)
N = 100
X = np.random.randn(N, 2)
# Create labels based on regions
y = np.zeros(N, dtype=int)
y[X[:, 0] > 0.5] = 1
y[X[:, 0] < -0.5] = 2

# Training
layer = Linear(2, 3)
lr = 0.1
losses = []

for epoch in range(200):
    logits = layer.forward(X)
    loss, dlogits = cross_entropy_loss(logits, y)
    layer.backward(dlogits)
    
    # SGD update
    layer.W -= lr * layer.dW
    layer.b -= lr * layer.db
    
    losses.append(loss)

# Plot training loss
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (Single Linear Layer)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Final accuracy
preds = np.argmax(layer.forward(X), axis=1)
acc = np.mean(preds == y)
print(f"Final loss: {losses[-1]:.4f}")
print(f"Accuracy:   {acc:.1%}")

## Summary

We now have the fundamental building blocks:
- **Linear layer** with forward and backward passes
- **ReLU** activation with gradient
- **Softmax + Cross-entropy** loss with combined gradient
- **Numerical gradient checking** to verify correctness
- A working **training loop** with SGD

Next notebook: **Embeddings and Data Preparation** - we'll build character-level embeddings and prepare our training data.